In [1]:
import pandas as pd

df = pd.read_csv('data/athlete_events.csv')
noc = pd.read_csv('data/noc_regions.csv')
df = df.merge(noc[['NOC', 'region']], on='NOC', how='left')
print(f'{len(df)} rows x {df.shape[1]} columns')
df.head()

271116 rows x 16 columns


,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal,region
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN,China
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN,China
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN,Denmark
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold,Denmark
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN,Netherlands


### Basic shape & types

In [2]:
print(df.dtypes)
print(f'\nUnique athletes: {df["ID"].nunique()}')
print(f'Unique events: {df["Event"].nunique()}')
print(f'Unique sports: {df["Sport"].nunique()}')
print(f'Unique NOCs: {df["NOC"].nunique()}')
print(f'Year range: {df["Year"].min()} - {df["Year"].max()}')
print(f'Seasons: {df["Season"].value_counts().to_dict()}')

ID          int64
Name          str
Sex           str
Age       float64
Height    float64
Weight    float64
Team          str
NOC           str
Games         str
Year        int64
Season        str
City          str
Sport         str
Event         str
Medal         str
region        str
dtype: object

Unique athletes: 135571
Unique events: 765
Unique sports: 66
Unique NOCs: 230
Year range: 1896 - 2016
Seasons: {'Summer': 222552, 'Winter': 48564}


### Missing data

In [3]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'count': missing, 'pct': missing_pct})[missing > 0]

,count,pct
Age,9474,3.49
Height,60171,22.19
Weight,62875,23.19
Medal,231333,85.33
region,370,0.14


### Numerical summaries (Age, Height, Weight)

In [4]:
df[['Age', 'Height', 'Weight']].describe()

,Age,Height,Weight
count,261642.000000,210945.000000,208241.000000
mean,25.556898,175.338970,70.702393
std,6.393561,10.518462,14.348020
min,10.000000,127.000000,25.000000
25%,21.000000,168.000000,60.000000
50%,24.000000,175.000000,70.000000
75%,28.000000,183.000000,79.000000
max,97.000000,226.000000,214.000000


### Medal distribution

In [5]:
print(df['Medal'].value_counts(dropna=False))
print(f'\nTotal medal entries: {df["Medal"].notna().sum()}')
print(f'Unique medalists: {df[df["Medal"].notna()]["ID"].nunique()}')

Medal
NaN       231333
Gold       13372
Bronze     13295
Silver     13116
Name: count, dtype: int64

Total medal entries: 39783
Unique medalists: 28251


### NOC regions lookup

In [6]:
print(f'noc_regions.csv: {noc.shape}')
print(f'Rows without region mapping after merge: {df["region"].isnull().sum()}')
noc.head(10)

noc_regions.csv: (230, 3)
Rows without region mapping after merge: 370


,NOC,region,notes
0,AFG,Afghanistan,NaN
1,AHO,Curacao,Netherlands Antilles
2,ALB,Albania,NaN
3,ALG,Algeria,NaN
4,AND,Andorra,NaN
5,ANG,Angola,NaN
6,ANT,Antigua,Antigua and Barbuda
7,ANZ,Australia,Australasia
8,ARG,Argentina,NaN
9,ARM,Armenia,NaN


### Number of events per Games over time

In [7]:
events_per_year = df.groupby(['Year', 'Season'])['Event'].nunique().reset_index(name='n_events')
events_per_year.sort_values('Year')

,Year,Season,n_events
0,1896,Summer,43
1,1900,Summer,90
2,1904,Summer,95
3,1906,Summer,74
4,1908,Summer,109
5,1912,Summer,107
6,1920,Summer,158
7,1924,Summer,131
8,1924,Winter,17
9,1928,Summer,122


### Top 15 countries by total medals

In [8]:
medals = df[df['Medal'].notna()]
medals.groupby('region')['Medal'].count().sort_values(ascending=False).head(15)

region
USA            5637
Russia         3947
Germany        3756
UK             2068
France         1777
Italy          1637
Sweden         1536
Canada         1352
Australia      1349
Hungary        1135
Netherlands    1040
Norway         1033
China           993
Japan           913
Finland         900
Name: Medal, dtype: int64